# 손실 함수와 평가 지표

이 노트북은 `src/nn/losses.py`의 손실 함수 3개와 `src/nn/metrics.py`의 평가 지표 3개를 직접 실행하고 시각화하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**목표**
- `cross_entropy`, `binary_cross_entropy`, `mse` 손실 함수와 각 `_grad` 함수를 검증한다.
- `accuracy`, `binary_accuracy`, `r2_score` 평가 지표를 검증한다.
- logit 입력 설계 원칙과 gradient 단순화 수식을 코드로 확인한다.
- 3가지 task의 손실 곡선을 시각화하여 비교한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.nn.losses import (
    cross_entropy, cross_entropy_grad,
    binary_cross_entropy, binary_cross_entropy_grad,
    mse, mse_grad,
)
from src.nn.metrics import accuracy, binary_accuracy, r2_score
from src.nn.activations import softmax, sigmoid

## 1. 개요

손실 함수는 모델 출력과 정답 레이블 사이의 오차를 단일 스칼라로 요약한다.
학습 루프는 이 스칼라를 최소화하는 방향으로 파라미터를 업데이트한다.

평가 지표는 손실과 별개로 모델 성능을 사람이 이해하기 쉬운 단위로 표현하며,
파라미터 업데이트에는 영향을 주지 않는다.

이 프로젝트의 task별 손실 함수와 평가 지표는 다음과 같다.

| task | 손실 함수 | 내부 activation | 평가 지표 |
|---|---|---|---|
| `multiclass` | `cross_entropy` | `softmax` | `accuracy` |
| `binary` | `binary_cross_entropy` | `sigmoid` | `binary_accuracy` |
| `regression` | `mse` | 없음 (identity) | `r2_score` |

**logit 입력 설계 원칙**: 손실 함수가 activation을 내부에서 처리하면 backward gradient 수식이 단순해진다.

$$
\frac{\partial L}{\partial z} = \frac{\hat{y} - y}{N}
$$

$z$는 logit, $\hat{y}$는 activation 적용 후 예측값, $y$는 정답, $N$은 배치 크기이다.

## 2. Cross Entropy (multiclass)

다중 클래스 분류에 사용한다. logit을 먼저 `softmax`로 확률로 변환한 뒤, 정답 클래스의 log 확률을 최대화한다.

$$
L_{\text{CE}} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{ic} \log(\hat{y}_{ic})
$$

gradient (softmax + CE 결합):

$$
\frac{\partial L_{\text{CE}}}{\partial z} = \frac{\hat{y} - y}{N}
$$

In [ ]:
np.random.seed(42)
N, C = 4, 10

logits = np.random.randn(N, C).astype(np.float32)
targets = np.eye(C, dtype=np.float32)[[0, 1, 2, 3]]

loss = cross_entropy(logits, targets)
grad = cross_entropy_grad(logits, targets)

print(f"logits shape:  {logits.shape}")
print(f"targets shape: {targets.shape}")
print(f"loss:  {loss:.4f}  (scalar)")
print(f"grad shape: {grad.shape}")
print(f"grad sum:   {grad.sum():.6f}  <- must be ~0 (softmax jacobian 성질)")

In [ ]:
# 완벽한 예측 시 loss가 최솟값에 가까워야 한다
# logit을 정답 클래스 방향으로 매우 크게 설정
logits_perfect = np.full((1, 10), -10.0, dtype=np.float32)
logits_perfect[0, 3] = 10.0  # 정답은 클래스 3
targets_perfect = np.eye(10, dtype=np.float32)[[3]]

logits_random = np.full((1, 10), 0.1, dtype=np.float32)  # 무작위 예측

loss_perfect = cross_entropy(logits_perfect, targets_perfect)
loss_random  = cross_entropy(logits_random, targets_perfect)

print(f"완벽한 예측 loss: {loss_perfect:.6f}  (매우 작아야 함)")
print(f"무작위 예측 loss: {loss_random:.6f}   (~log(C) = {np.log(10):.4f})")

## 3. Binary Cross Entropy (binary)

이진 분류에 사용한다. logit을 `sigmoid`로 변환한 뒤, 정답 레이블에 따라 log 확률을 최대화한다.

$$
L_{\text{BCE}} = -\frac{1}{N} \sum_{i=1}^{N} \bigl[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \bigr]
$$

gradient (sigmoid + BCE 결합):

$$
\frac{\partial L_{\text{BCE}}}{\partial z} = \frac{\hat{y} - y}{N}
$$

In [ ]:
np.random.seed(42)
N = 4

logits = np.random.randn(N, 1).astype(np.float32)
targets = np.array([[1.], [0.], [1.], [0.]], dtype=np.float32)

loss = binary_cross_entropy(logits, targets)
grad = binary_cross_entropy_grad(logits, targets)

print(f"logits shape:  {logits.shape}")
print(f"targets shape: {targets.shape}")
print(f"loss:  {loss:.4f}  (scalar)")
print(f"grad shape: {grad.shape}")
print(f"grad = (sigmoid(logits) - targets) / N:")
expected_grad = (sigmoid(logits) - targets) / N
print(f"  computed: {grad.ravel().round(4)}")
print(f"  expected: {expected_grad.ravel().round(4)}")
print(f"  match: {np.allclose(grad, expected_grad)}")

## 4. Mean Squared Error (regression)

회귀에 사용한다. activation이 없으므로 logit이 곧 예측값이며, 예측값과 정답의 차이 제곱의 평균을 최소화한다.

$$
L_{\text{MSE}} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2
$$

$$
\frac{\partial L_{\text{MSE}}}{\partial \hat{y}} = \frac{2(\hat{y} - y)}{N}
$$

In [ ]:
np.random.seed(42)
N = 4

# regression: 정답은 label/9.0 ∈ [0, 1]
preds   = np.array([[0.1], [0.6], [0.8], [0.3]], dtype=np.float32)
targets = np.array([[0.1], [0.5], [0.9], [0.3]], dtype=np.float32)

loss = mse(preds, targets)
grad = mse_grad(preds, targets)

print(f"preds   shape: {preds.shape}")
print(f"targets shape: {targets.shape}")
print(f"loss:  {loss:.6f}  (scalar)")
print(f"grad shape: {grad.shape}")
print(f"grad = 2*(preds - targets)/N:")
expected_grad = 2.0 * (preds - targets) / N
print(f"  computed: {grad.ravel().round(4)}")
print(f"  expected: {expected_grad.ravel().round(4)}")
print(f"  match: {np.allclose(grad, expected_grad)}")

# 동일한 입력 시 loss = 0
print(f"\nmse(x, x) = {mse(preds, preds):.6f}  <- must be 0")

## 5. 평가 지표

### 5.1. Accuracy (multiclass)

$$
\text{Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}\bigl[\arg\max(\hat{z}_i) = \arg\max(y_i)\bigr]
$$

`softmax`는 단조 증가 함수이므로 logit에 직접 `argmax`를 적용해도 결과가 동일하다.

### 5.2. Binary Accuracy (binary)

$$
\text{Binary Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}\bigl[\mathbf{1}[\hat{z}_i \geq 0] = y_i\bigr]
$$

`sigmoid(x) >= 0.5`는 `x >= 0`과 동치이므로 sigmoid를 계산하지 않고 logit 부호만 확인한다.

### 5.3. R2 Score (regression)

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}, \quad SS_{res} = \sum(y - \hat{y})^2, \quad SS_{tot} = \sum(y - \bar{y})^2
$$

In [ ]:
# accuracy
logits_mc = np.array([[2.0, 0.5, 0.1], [0.1, 0.5, 2.0], [1.0, 2.0, 0.0]], dtype=np.float32)
targets_mc = np.array([[1., 0., 0.], [0., 0., 1.], [0., 1., 0.]], dtype=np.float32)
acc = accuracy(logits_mc, targets_mc)
print(f"accuracy (완벽한 예측): {acc:.4f}  <- 1.0 기대")

# binary_accuracy
logits_b = np.array([[1.0], [-1.0], [0.5], [-0.5]], dtype=np.float32)
targets_b = np.array([[1.], [0.], [1.], [0.]], dtype=np.float32)
bacc = binary_accuracy(logits_b, targets_b)
print(f"binary_accuracy (완벽한 예측): {bacc:.4f}  <- 1.0 기대")

# r2_score
preds_perfect = np.array([[0.1], [0.5], [0.8]], dtype=np.float32)
targets_r = np.array([[0.1], [0.5], [0.8]], dtype=np.float32)
r2 = r2_score(preds_perfect, targets_r)
print(f"r2_score (완벽한 예측): {r2:.4f}  <- 1.0 기대")

# r2_score: 평균 예측 시 ≈ 0
preds_mean = np.full_like(targets_r, targets_r.mean())
r2_mean = r2_score(preds_mean, targets_r)
print(f"r2_score (평균 예측):   {r2_mean:.4f}  <- 0.0 기대")

## 6. 손실 곡선 시각화

예측이 정답에서 멀어질수록 각 loss가 어떻게 증가하는지 시각화하여 비교한다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# -- Cross Entropy: 정답 클래스 logit을 변화시키며 loss 관찰
logit_vals = np.linspace(-3, 3, 100)
ce_losses = []
for v in logit_vals:
    # 정답=클래스0, logit[0]=v, 나머지=0
    lg = np.array([[v, 0.0, 0.0, 0.0, 0.0]])
    tg = np.array([[1., 0., 0., 0., 0.]])
    ce_losses.append(cross_entropy(lg, tg))
axes[0].plot(logit_vals, ce_losses, color='steelblue', linewidth=2)
axes[0].axvline(0, color='gray', linewidth=0.5, linestyle='--')
axes[0].set_title("Cross Entropy\n(정답 클래스 logit 변화)")
axes[0].set_xlabel("logit[0] (정답 클래스)")
axes[0].set_ylabel("loss")
axes[0].grid(alpha=0.3)

# -- BCE: logit을 변화시키며 loss 관찰
bce_losses_1 = []
bce_losses_0 = []
for v in logit_vals:
    lg = np.array([[v]])
    bce_losses_1.append(binary_cross_entropy(lg, np.array([[1.0]])))
    bce_losses_0.append(binary_cross_entropy(lg, np.array([[0.0]])))
axes[1].plot(logit_vals, bce_losses_1, color='tomato',    linewidth=2, label='target=1')
axes[1].plot(logit_vals, bce_losses_0, color='steelblue', linewidth=2, label='target=0')
axes[1].axvline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_title("Binary Cross Entropy\n(logit 변화)")
axes[1].set_xlabel("logit")
axes[1].set_ylabel("loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

# -- MSE: 예측값을 변화시키며 loss 관찰
pred_vals = np.linspace(-1, 2, 100)
mse_losses = []
for v in pred_vals:
    p = np.array([[v]])
    t = np.array([[0.5]])
    mse_losses.append(mse(p, t))
axes[2].plot(pred_vals, mse_losses, color='forestgreen', linewidth=2)
axes[2].axvline(0.5, color='gray', linewidth=0.5, linestyle='--', label='target=0.5')
axes[2].set_title("MSE\n(pred 변화, target=0.5)")
axes[2].set_xlabel("pred")
axes[2].set_ylabel("loss")
axes[2].legend()
axes[2].grid(alpha=0.3)

fig.suptitle("Loss Functions", fontsize=13)
fig.tight_layout()
plt.show()

## 7. 검증

In [ ]:
# cross_entropy
N, C = 4, 10
logits = np.random.randn(N, C).astype(np.float32)
targets = np.eye(C, dtype=np.float32)[np.arange(N)]
loss = cross_entropy(logits, targets)
grad = cross_entropy_grad(logits, targets)
assert np.ndim(loss) == 0, "loss must be scalar"
assert grad.shape == logits.shape, "grad shape mismatch"
assert abs(grad.sum()) < 1e-5, "cross_entropy_grad sum must be ~0"

# binary_cross_entropy
logits_b = np.random.randn(N, 1).astype(np.float32)
targets_b = (np.random.rand(N, 1) > 0.5).astype(np.float32)
loss_b = binary_cross_entropy(logits_b, targets_b)
grad_b = binary_cross_entropy_grad(logits_b, targets_b)
assert np.ndim(loss_b) == 0, "loss_b must be scalar"
assert grad_b.shape == logits_b.shape, "grad_b shape mismatch"

# mse
preds = np.random.rand(N, 1).astype(np.float32)
targets_r = np.random.rand(N, 1).astype(np.float32)
loss_r = mse(preds, targets_r)
grad_r = mse_grad(preds, targets_r)
assert np.ndim(loss_r) == 0, "loss_r must be scalar"
assert grad_r.shape == preds.shape, "grad_r shape mismatch"
assert mse(preds, preds) < 1e-10, "mse(x, x) must be ~0"

# accuracy
logits_mc = np.array([[2.0, 0.1], [0.1, 2.0]], dtype=np.float32)
targets_mc = np.array([[1., 0.], [0., 1.]], dtype=np.float32)
assert accuracy(logits_mc, targets_mc) == 1.0, "perfect prediction must be 1.0"

# binary_accuracy
logits_b2 = np.array([[1.0], [-1.0]], dtype=np.float32)
targets_b2 = np.array([[1.], [0.]], dtype=np.float32)
assert binary_accuracy(logits_b2, targets_b2) == 1.0, "perfect binary prediction must be 1.0"

# r2_score
p = np.array([[1.0], [2.0], [3.0]], dtype=np.float32)
assert abs(r2_score(p, p) - 1.0) < 1e-5, "r2_score(x, x) must be 1.0"

print("all loss/metric validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 함수 | 입력 | 출력 | 핵심 특징 |
|---|---|---|---|
| `cross_entropy` | logits `(N, C)`, targets one-hot `(N, C)` | scalar | softmax 내부 처리, gradient sum ≈ 0 |
| `binary_cross_entropy` | logits `(N, 1)`, targets `(N, 1)` | scalar | sigmoid 내부 처리, log clip 적용 |
| `mse` | preds `(N, 1)`, targets `(N, 1)` | scalar | activation 없음, identity 출력 |
| `accuracy` | logits `(N, C)`, targets one-hot | scalar `[0, 1]` | softmax 불필요 |
| `binary_accuracy` | logits `(N, 1)`, targets `(N, 1)` | scalar `[0, 1]` | sigmoid 불필요 (`x >= 0`) |
| `r2_score` | preds `(N, 1)`, targets `(N, 1)` | scalar `(-∞, 1]` | 완벽=1.0, 평균예측=0.0 |

**핵심 설계 원칙**
- `cross_entropy`와 `binary_cross_entropy`는 activation을 내부에서 처리하므로 gradient가 `(activation(logits) - targets) / N`으로 단순화된다.
- `accuracy`와 `binary_accuracy`는 logit에 직접 적용해도 activation 후와 동일한 결과를 반환한다.